# Part 3 — SERP-Level Data

**Endpoint:** `POST /v1/serp/tasks` (async: submit → poll → collect)  
**Cost:** 10 credits/keyword  
**Note:** Snapshot only — labeled cross-sectional.  
**Outputs:** `data/processed/serp_results.csv`, `serp_brand_presence.csv`, `keyword_list.csv`

In [1]:
import sys, json, time
from pathlib import Path
from datetime import date
import requests
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from config import (
    API_BASE, HEADERS, RATE_LIMIT_DELAY,
    ALL_DOMAINS, DOMAIN_META, SOURCE,
    DATA_RAW, DATA_PROCESSED, DATA_KEYWORDS
)

SNAPSHOT_DATE = str(date.today())

def api_get(path, params=None):
    url = f"{API_BASE}{path}"
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

def api_post(path, json_body=None):
    url = f"{API_BASE}{path}"
    resp = requests.post(url, headers=HEADERS, json=json_body, timeout=30)
    time.sleep(RATE_LIMIT_DELAY)
    return resp

In [2]:
# ── Subscription / credit check ────────────────────────────────────────────────
sub = api_get("/v1/account/subscription").json().get("subscription_info", {})
status  = sub.get("status", "unknown")
expires = sub.get("expiraton_date", "unknown")
units   = sub.get("units_left", 0)

print(f"Subscription status : {status}")
print(f"Expiration date     : {expires}")
print(f"Units left          : {units}")

if status != "active":
    print(f"\n⚠️  Subscription is '{status}' — SERP tasks will fail (402).")
    print("   Renew at https://seranking.com/api then re-run this notebook.")

Subscription status : active
Expiration date     : 2026-11-10 19:25:58
Units left          : 99818559.0


## 1. Load keyword lists

In [3]:
def load_keywords(filepath):
    lines = Path(filepath).read_text().splitlines()
    return [l.strip() for l in lines if l.strip() and not l.strip().startswith("#")]

bev_t1 = load_keywords(DATA_KEYWORDS / "beverages_tier1.txt")
bev_t2 = load_keywords(DATA_KEYWORDS / "beverages_tier2.txt")
ff_t1  = load_keywords(DATA_KEYWORDS / "fastfood_tier1.txt")
ff_t2  = load_keywords(DATA_KEYWORDS / "fastfood_tier2.txt")

keywords = {
    "beverages": {"tier1": bev_t1, "tier2": bev_t2, "all": bev_t1 + bev_t2},
    "fast_food":  {"tier1": ff_t1,  "tier2": ff_t2,  "all": ff_t1  + ff_t2}
}

for cat, kw in keywords.items():
    print(f"{cat}: {len(kw['tier1'])} Tier-1  +  {len(kw['tier2'])} Tier-2  =  {len(kw['all'])} total")

total_kw = sum(len(v["all"]) for v in keywords.values())
print(f"\nTotal SERP tasks: {total_kw} keywords × 10 credits = ~{total_kw * 10:,} credits")

# Save keyword metadata
kw_rows = [
    {"category": cat, "tier": tier, "keyword": kw}
    for cat, kw_data in keywords.items()
    for tier in ["tier1", "tier2"]
    for kw in kw_data[tier]
]
pd.DataFrame(kw_rows).to_csv(DATA_PROCESSED / "keyword_list.csv", index=False)
print(f"Keyword list saved: {DATA_PROCESSED / 'keyword_list.csv'}")

beverages: 40 Tier-1  +  72 Tier-2  =  112 total
fast_food: 55 Tier-1  +  81 Tier-2  =  136 total

Total SERP tasks: 248 keywords × 10 credits = ~2,480 credits
Keyword list saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/keyword_list.csv


## 2. Discover SERP engine ID (Google US Desktop)

In [4]:
# Classic endpoint parameters for Google US Desktop
# POST /v1/serp/classic/tasks with these fixed params:
SERP_PARAMS = {
    "search_engine": "google",
    "device": "desktop",
    "language_code": "en",
    "google_ads_location_id": 2840,   # United States
}
print("SERP classic endpoint params:", SERP_PARAMS)

SERP classic endpoint params: {'search_engine': 'google', 'device': 'desktop', 'language_code': 'en', 'google_ads_location_id': 2840}


## 3. Submit SERP tasks (batched, 50 keywords/request)

In [5]:
def submit_serp_tasks(keyword_list):
    """Submit one keyword at a time via POST /v1/serp/classic/tasks (returns 201)."""
    submitted = []
    for i, kw in enumerate(keyword_list):
        resp = api_post("/v1/serp/classic/tasks", json_body={
            "query": kw,
            **SERP_PARAMS,
        })
        if resp.status_code in (200, 201):
            task_list = resp.json()
            if isinstance(task_list, list):
                submitted.extend(task_list)
            elif isinstance(task_list, dict):
                submitted.append(task_list)
        else:
            print(f"  ERROR {resp.status_code} for '{kw}': {resp.text[:100]}")
        if (i + 1) % 25 == 0:
            print(f"  Submitted {i+1}/{len(keyword_list)} keywords...")
    return submitted

all_tasks = {}
for category, kw_data in keywords.items():
    print(f"\n--- {category} ({len(kw_data['all'])} keywords) ---")
    tasks = submit_serp_tasks(kw_data["all"])
    all_tasks[category] = tasks
    print(f"  Submitted: {len(tasks)} tasks")

raw_tasks_path = DATA_RAW / "serp_task_ids.json"
raw_tasks_path.write_text(json.dumps(all_tasks, indent=2))
total_submitted = sum(len(v) for v in all_tasks.values())
print(f"\nTotal tasks submitted: {total_submitted} | Saved: {raw_tasks_path}")


--- beverages (112 keywords) ---


  Submitted 25/112 keywords...


  Submitted 50/112 keywords...


  Submitted 75/112 keywords...


  Submitted 100/112 keywords...


...

  Submitted 50/136 keywords...


  Submitted 75/136 keywords...


  Submitted 100/136 keywords...


  Submitted 125/136 keywords...


  Submitted: 136 tasks

Total tasks submitted: 248 | Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/raw/serp_task_ids.json


## 4. Poll for results

In [6]:
def fetch_serp_results(task_id, max_retries=12, retry_delay=15):
    """Poll GET /v1/serp/classic/tasks?task_id= until results arrive."""
    for attempt in range(max_retries):
        resp = api_get("/v1/serp/classic/tasks", params={"task_id": task_id})
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, dict):
                if data.get("status") == "processing":
                    time.sleep(retry_delay)
                    continue
                # Completed — items array contains SERP results
                return data.get("items") or []
            elif isinstance(data, list):
                return data
        time.sleep(retry_delay)
    print(f"    Task {task_id}: timed out")
    return None

# Wait for tasks to process before polling
if total_submitted > 0:
    print("Waiting 30s for tasks to process...")
    time.sleep(30)

all_serp_results = {}
failed_tasks     = []

for category, tasks in all_tasks.items():
    print(f"\n--- Polling {category} ({len(tasks)} tasks) ---")
    for task_info in tasks:
        task_id = task_info.get("id") or task_info.get("task_id")
        query   = task_info.get("query", "")
        if not task_id:
            continue
        results = fetch_serp_results(task_id)
        if results:
            all_serp_results[str(task_id)] = {
                "query": query, "category": category,
                "snapshot_date": SNAPSHOT_DATE, "results": results
            }
        else:
            failed_tasks.append({"task_id": task_id, "query": query, "category": category})

print(f"\nSuccessful: {len(all_serp_results)} | Failed: {len(failed_tasks)}")
(DATA_RAW / "serp_results_raw.json").write_text(json.dumps(all_serp_results, indent=2))
print(f"Raw saved: {DATA_RAW / 'serp_results_raw.json'}")

Waiting 30s for tasks to process...



--- Polling beverages (112 tasks) ---



--- Polling fast_food (136 tasks) ---



Successful: 248 | Failed: 0


Raw saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/raw/serp_results_raw.json


## 5. Parse results + compute brand presence

In [7]:
BRAND_DOMAINS = {
    "beverages": {
        "coca-cola.com":   ["coca-cola.com"],
        "pepsi.com":       ["pepsi.com"],
        "drpepper.com":    ["drpepper.com"],
        "drinkolipop.com": ["drinkolipop.com", "olipop.com"],
    },
    "fast_food": {
        "mcdonalds.com":  ["mcdonalds.com"],
        "wendys.com":     ["wendys.com"],
        "shakeshack.com": ["shakeshack.com"],
        "fiveguys.com":   ["fiveguys.com"],
    }
}

rows = []
for task_id, task_data in all_serp_results.items():
    for result in (task_data["results"] if isinstance(task_data["results"], list) else []):
        if not isinstance(result, dict):
            continue
        rows.append({
            "task_id":       task_id,
            "query":         task_data["query"],
            "category":      task_data["category"],
            "snapshot_date": task_data["snapshot_date"],
            # classic endpoint uses rank_absolute; fall back to position/rank
            "position":      result.get("rank_absolute") or result.get("position") or result.get("rank"),
            "url":           result.get("url")   or result.get("link"),
            "title":         result.get("title"),
            "snippet":       result.get("description") or result.get("snippet"),
            "domain":        result.get("domain") or result.get("host"),
            "serp_features": json.dumps(result.get("serp_features", [])),
        })

df_serp = pd.DataFrame(rows)

if df_serp.empty:
    print("⚠️  No SERP results — subscription may be expired.")
    df_brand = pd.DataFrame()
else:
    df_serp["position"] = pd.to_numeric(df_serp["position"], errors="coerce")
    print(f"SERP rows: {len(df_serp)}")
    display(df_serp.head(5))

    # Brand presence metrics
    def url_has_domain(url, domains):
        return bool(url) and any(d in str(url).lower() for d in domains)

    bp_rows = []
    for category, brand_map in BRAND_DOMAINS.items():
        cat_df = df_serp[df_serp["category"] == category]
        for query, q_df in cat_df.groupby("query"):
            top10 = q_df[q_df["position"] <= 10]
            top20 = q_df[q_df["position"] <= 20]
            for brand_domain, bdlist in brand_map.items():
                brand_all  = q_df[q_df["url"].apply(lambda u: url_has_domain(u, bdlist))]
                brand_t10  = top10[top10["url"].apply(lambda u: url_has_domain(u, bdlist))]
                brand_t20  = top20[top20["url"].apply(lambda u: url_has_domain(u, bdlist))]
                bp_rows.append({
                    "query": query, "category": category, "brand_domain": brand_domain,
                    "size_tier": DOMAIN_META.get(brand_domain, {}).get("size_tier", "unknown"),
                    "snapshot_date": q_df["snapshot_date"].iloc[0],
                    "brand_present_top10":        int(len(brand_t10) > 0),
                    "best_rank_brand":             brand_all["position"].min() if not brand_all.empty else None,
                    "count_brand_results_top10":   len(brand_t10),
                    "count_brand_results_top20":   len(brand_t20),
                })
    df_brand = pd.DataFrame(bp_rows)
    print(f"Brand presence rows: {len(df_brand)}")
    display(df_brand.head(8))

SERP rows: 23059


,task_id,query,category,snapshot_date,position,url,title,snippet,domain,serp_features
0,32411998,coca cola,beverages,2026-03-02,1,https://www.coca-cola.com/us/en,Home Page | Coca-Cola US,Explore ways you can be closer to the ones you...,www.coca-cola.com,[]
1,32411998,coca cola,beverages,2026-03-02,2,https://www.coca-colacompany.com/,The Coca-Cola Company: Refresh the World. Make...,We are here to refresh the world and make a di...,www.coca-colacompany.com,[]
2,32411998,coca cola,beverages,2026-03-02,3,https://en.wikipedia.org/wiki/Coca-Cola,Coca-Cola,"Coca-Cola, or Coke, is a cola soft drink manuf...",en.wikipedia.org,[]
3,32411998,coca cola,beverages,2026-03-02,4,https://www.coca-colacompany.com/careers,Careers,Work with Us. The Coca‑Cola Company is a globa...,www.coca-colacompany.com,[]
4,32411998,coca cola,beverages,2026-03-02,6,https://www.worldofcoca-cola.com/,"World of Coca-Cola: Atlanta Attraction, Museum...","Welcome to the World of Coca-Cola, an Atlanta ...",www.worldofcoca-cola.com,[]


Brand presence rows: 992


,query,category,brand_domain,size_tier,snapshot_date,brand_present_top10,best_rank_brand,count_brand_results_top10,count_brand_results_top20
0,artificial sweetener soda,beverages,coca-cola.com,large,2026-03-02,0,NaN,0,0
1,artificial sweetener soda,beverages,pepsi.com,large,2026-03-02,0,NaN,0,0
2,artificial sweetener soda,beverages,drpepper.com,large,2026-03-02,0,NaN,0,0
3,artificial sweetener soda,beverages,drinkolipop.com,small,2026-03-02,0,NaN,0,0
4,artisanal soda,beverages,coca-cola.com,large,2026-03-02,0,NaN,0,0
5,artisanal soda,beverages,pepsi.com,large,2026-03-02,0,NaN,0,0
6,artisanal soda,beverages,drpepper.com,large,2026-03-02,0,NaN,0,0
7,artisanal soda,beverages,drinkolipop.com,small,2026-03-02,0,NaN,0,0


## 6. Save & summarize

In [8]:
if not df_serp.empty:
    df_serp.to_csv(DATA_PROCESSED / "serp_results.csv", index=False)
    print(f"Saved: {DATA_PROCESSED / 'serp_results.csv'}")

if not df_brand.empty:
    df_brand.to_csv(DATA_PROCESSED / "serp_brand_presence.csv", index=False)
    print(f"Saved: {DATA_PROCESSED / 'serp_brand_presence.csv'}")

    print("\n=== Top-10 Presence Rate by Brand ===")
    summary = df_brand.groupby(["category","brand_domain","size_tier"]).agg(
        keywords_total=("query","count"),
        top10_presence_pct=("brand_present_top10","mean"),
        median_best_rank=("best_rank_brand","median")
    ).reset_index()
    summary["top10_presence_pct"] = (summary["top10_presence_pct"] * 100).round(1)
    display(summary.sort_values(["category","top10_presence_pct"], ascending=[True, False]))
    print(f"\n⚠️  Cross-sectional snapshot ({SNAPSHOT_DATE}). No historical comparison available.")

Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/serp_results.csv
Saved: /Users/vince/Projects/BITLab/GEO-exploration/seranking/data/processed/serp_brand_presence.csv

=== Top-10 Presence Rate by Brand ===


,category,brand_domain,size_tier,keywords_total,top10_presence_pct,median_best_rank
0,beverages,coca-cola.com,large,112,16.1,23.0
1,beverages,drinkolipop.com,small,112,11.6,18.0
2,beverages,drpepper.com,large,112,8.0,5.0
3,beverages,pepsi.com,large,112,5.4,35.5
5,fast_food,mcdonalds.com,large,136,23.5,10.0
7,fast_food,wendys.com,large,136,18.4,17.0
6,fast_food,shakeshack.com,small,136,14.0,15.0
4,fast_food,fiveguys.com,small,136,11.0,32.0



⚠️  Cross-sectional snapshot (2026-03-02). No historical comparison available.
